In [ ]:
!pip install numpy scipy scikit-learn opencv-python matplotlib tqdm
import os
import cv2
import numpy as np
from scipy.linalg import eigh
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import random

class FaceImageLoader:
    def __init__(self, root_path, target_size=(100, 100), max_persons=None, max_images_per_person=10, random_seed=42):
        self.root_path = root_path
        self.target_size = target_size
        self.max_persons = max_persons
        self.max_images_per_person = max_images_per_person
        self.random_seed = random_seed
        self.image_paths = []
        self.labels = []
        self.label_dict = {}
        self.label_names = []

    def load_metadata(self):
        random.seed(self.random_seed)
        person_folders = [d for d in sorted(os.listdir(self.root_path))
                         if os.path.isdir(os.path.join(self.root_path, d))]

        if self.max_persons and len(person_folders) > self.max_persons:
            person_folders = random.sample(person_folders, self.max_persons)

        for person_id, person in enumerate(tqdm(person_folders, desc="Loading data")):
            self.label_dict[person] = person_id
            self.label_names.append(person)

            person_path = os.path.join(self.root_path, person)
            all_images = []

            for root, _, files in os.walk(person_path):
                for file in files:
                    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        all_images.append(os.path.join(root, file))

            if self.max_images_per_person and len(all_images) > self.max_images_per_person:
                all_images = random.sample(all_images, self.max_images_per_person)

            for img_path in all_images:
                self.image_paths.append(img_path)
                self.labels.append(person_id)

        self.image_paths = np.array(self.image_paths)
        self.labels = np.array(self.labels)

    def load_batch(self, paths, labels=None):
        batch = []
        batch_labels = [] if labels is not None else None

        for i, img_path in enumerate(paths):
            try:
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                img = cv2.resize(img, self.target_size)
                img = img.astype(np.float32) / 255.0
                batch.append(img)

                if labels is not None:
                    batch_labels.append(labels[i])
            except:
                continue

        return np.array(batch), np.array(batch_labels) if batch_labels is not None else None

class CSA:
    def __init__(self, m_prime=50, n_prime=50, max_iter=10, reg_param=1e-3):
        self.m_prime = m_prime
        self.n_prime = n_prime
        self.max_iter = max_iter
        self.reg_param = reg_param
        self.U = None
        self.V = None
        self.overall_mean = None
        self.class_means = {}
        self.iteration_history = []

    def _compute_statistics(self, data_loader, indices):
        total_sum = np.zeros(data_loader.target_size, dtype=np.float32)
        total_count = 0
        class_sums = defaultdict(lambda: np.zeros(data_loader.target_size, dtype=np.float32))
        class_counts = defaultdict(int)

        batch_size = 100
        for i in range(0, len(indices), batch_size):
            batch_indices = indices[i:i+batch_size]
            X, y = data_loader.load_batch(
                data_loader.image_paths[batch_indices],
                data_loader.labels[batch_indices]
            )

            if X is None:
                continue

            total_sum += np.sum(X, axis=0)
            total_count += len(X)

            for c in np.unique(y):
                class_mask = (y == c)
                class_sums[c] += np.sum(X[class_mask], axis=0)
                class_counts[c] += np.sum(class_mask)

        self.overall_mean = total_sum / total_count
        self.class_means = {c: class_sums[c]/class_counts[c] for c in class_sums}

    def _update_parameters(self, data_loader, train_indices):
        unique, counts = np.unique(data_loader.labels[train_indices], return_counts=True)
        class_counts = dict(zip(unique, counts))

        # Update V
        F = np.zeros((self.V.shape[0], self.V.shape[0]))
        for c, mean in self.class_means.items():
            diff = mean - self.overall_mean
            F += class_counts[c] * diff.T @ self.U @ self.U.T @ diff
        F += self.reg_param * np.eye(F.shape[0])
        _, V = eigh(F)
        self.V = V[:, -self.n_prime:]

        # Update U
        G = np.zeros((self.U.shape[0], self.U.shape[0]))
        for c, mean in self.class_means.items():
            diff = mean - self.overall_mean
            G += class_counts[c] * diff @ self.V @ self.V.T @ diff.T
        G += self.reg_param * np.eye(G.shape[0])
        _, U = eigh(G)
        self.U = U[:, -self.m_prime:]

    def fit(self, data_loader, train_indices, val_indices):
        self._compute_statistics(data_loader, train_indices)
        m, n = self.overall_mean.shape
        self.U = np.eye(m, self.m_prime)
        self.V = np.eye(n, self.n_prime)

        for iter_num in range(1, self.max_iter+1):
            self._update_parameters(data_loader, train_indices)

            train_features = self.transform(data_loader, train_indices)
            val_features = self.transform(data_loader, val_indices)

            result = self._evaluate_performance(
                train_features, data_loader.labels[train_indices],
                val_features, data_loader.labels[val_indices]
            )

            self.iteration_history.append({
                'iteration': iter_num,
                'rank1': result['rank1'],
                'rank5': result['rank5']
            })

    def transform(self, data_loader, indices):
        projected = []
        batch_size = 100
        for i in range(0, len(indices), batch_size):
            batch_indices = indices[i:i+batch_size]
            X, _ = data_loader.load_batch(data_loader.image_paths[batch_indices])
            if X is not None:
                projected.append(np.stack([self.U.T @ x @ self.V for x in X]))
        return np.concatenate(projected) if projected else np.array([])

    def _evaluate_performance(self, gallery_features, gallery_labels, probe_features, probe_labels):
        gallery_flat = gallery_features.reshape(len(gallery_features), -1)
        probe_flat = probe_features.reshape(len(probe_features), -1)

        knn = KNeighborsClassifier(n_neighbors=5)
        knn.fit(gallery_flat, gallery_labels)

        distances, indices = knn.kneighbors(probe_flat)
        rank1 = np.mean(probe_labels == gallery_labels[indices[:, 0]])
        rank5 = np.mean([probe_labels[i] in gallery_labels[indices[i]] for i in range(len(probe_labels))])

        return {
            'rank1': 100 * rank1,
            'rank5': 100 * rank5
        }

class DATER:
    def __init__(self, n_components=100, max_iter=5):
        self.n_components = n_components
        self.max_iter = max_iter
        self.projection = None
        self.mean = None
        self.iteration_history = []

    def fit(self, data_loader, train_indices, val_indices):
        X_train, y_train = data_loader.load_batch(
            data_loader.image_paths[train_indices],
            data_loader.labels[train_indices]
        )
        X_val, y_val = data_loader.load_batch(
            data_loader.image_paths[val_indices],
            data_loader.labels[val_indices]
        )

        # Flatten images
        X_train = X_train.reshape(len(X_train), -1)
        X_val = X_val.reshape(len(X_val), -1)

        # Initialize with PCA
        self.mean = np.mean(X_train, axis=0)
        X_centered = X_train - self.mean
        cov = np.cov(X_centered.T)
        _, eig_vecs = eigh(cov, subset_by_index=[cov.shape[0]-self.n_components, cov.shape[0]-1])
        self.projection = eig_vecs.T

        for iter_num in range(1, self.max_iter+1):
            # DATER optimization would go here
            # For now using PCA as baseline

            # Evaluate
            train_proj = (X_train - self.mean) @ self.projection.T
            val_proj = (X_val - self.mean) @ self.projection.T

            result = self._evaluate_performance(train_proj, y_train, val_proj, y_val)

            self.iteration_history.append({
                'iteration': iter_num,
                'rank1': result['rank1'],
                'rank5': result['rank5']
            })

    def _evaluate_performance(self, gallery_features, gallery_labels, probe_features, probe_labels):
        knn = KNeighborsClassifier(n_neighbors=5)
        knn.fit(gallery_features, gallery_labels)

        distances, indices = knn.kneighbors(probe_features)
        rank1 = np.mean(probe_labels == gallery_labels[indices[:, 0]])
        rank5 = np.mean([probe_labels[i] in gallery_labels[indices[i]] for i in range(len(probe_labels))])

        return {
            'rank1': 100 * rank1,
            'rank5': 100 * rank5
        }

def plot_average_faces(data_loader, n_persons=5):
    sns.set_style("white")
    unique_labels = np.unique(data_loader.labels)
    selected_labels = np.random.choice(unique_labels, size=n_persons, replace=False)

    fig, axes = plt.subplots(1, n_persons, figsize=(15, 3))

    for i, label in enumerate(selected_labels):
        mask = data_loader.labels == label
        images, _ = data_loader.load_batch(data_loader.image_paths[mask])
        avg_face = np.mean(images, axis=0)

        axes[i].imshow(avg_face, cmap='gray')
        axes[i].set_title(f"ID: {label}")
        axes[i].axis('off')

    plt.suptitle(f"Average Faces of {n_persons} Random Subjects")
    plt.tight_layout()
    plt.show()

def plot_iteration_curves(csa_history, dater_history):
    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Get metrics from last fold
    csa_iter = [x['iteration'] for x in csa_history[-1]]
    csa_rank1 = [x['rank1'] for x in csa_history[-1]]
    csa_rank5 = [x['rank5'] for x in csa_history[-1]]

    dater_iter = [x['iteration'] for x in dater_history[-1]]
    dater_rank1 = [x['rank1'] for x in dater_history[-1]]
    dater_rank5 = [x['rank5'] for x in dater_history[-1]]

    # Rank-1 plot
    ax1.plot(csa_iter, csa_rank1, 'o-', label='CSA+DATER')
    ax1.plot(dater_iter, dater_rank1, 's-', label='DATER')
    ax1.set_title('Rank-1 Accuracy vs Iterations')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Accuracy (%)')
    ax1.legend()
    ax1.grid(True)

    # Rank-5 plot
    ax2.plot(csa_iter, csa_rank5, 'o-', label='CSA+DATER')
    ax2.plot(dater_iter, dater_rank5, 's-', label='DATER')
    ax2.set_title('Rank-5 Accuracy vs Iterations')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

def create_performance_table(csa_history, dater_history):
    # Get final metrics from all folds
    csa_final_rank1 = [fold[-1]['rank1'] for fold in csa_history]
    csa_final_rank5 = [fold[-1]['rank5'] for fold in csa_history]

    dater_final_rank1 = [fold[-1]['rank1'] for fold in dater_history]
    dater_final_rank5 = [fold[-1]['rank5'] for fold in dater_history]

    # Create dataframe
    data = {
        'Method': ['CSA+DATER']*len(csa_history) + ['DATER']*len(dater_history),
        'Fold': list(range(1, len(csa_history)+1)) + list(range(1, len(dater_history)+1)),
        'Rank-1': csa_final_rank1 + dater_final_rank1,
        'Rank-5': csa_final_rank5 + dater_final_rank5
    }
    df = pd.DataFrame(data)

    # Create summary table
    summary = df.groupby('Method').agg({
        'Rank-1': ['mean', 'std'],
        'Rank-5': ['mean', 'std']
    }).round(1)

    # Plot table
    fig, ax = plt.subplots(figsize=(8, 2))
    ax.axis('off')
    table = ax.table(
        cellText=summary.values,
        rowLabels=summary.index,
        colLabels=['Rank-1 (Mean)', 'Rank-1 (Std)', 'Rank-5 (Mean)', 'Rank-5 (Std)'],
        loc='center',
        cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    plt.title("Final Performance Comparison", pad=20)
    plt.show()

    return df, summary

def main():
    # Initialize data loader
    loader = FaceImageLoader("/content/aligned_images_DB")
    loader.load_metadata()

    # Prepare cross-validation
    n_splits = 10  # Using 10 folds for demonstration
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    csa_history = []
    dater_history = []

    for train_idx, val_idx in skf.split(loader.image_paths, loader.labels):
        # Train CSA+DATER
        csa = CSA(max_iter=5)
        csa.fit(loader, train_idx, val_idx)
        csa_history.append(csa.iteration_history)

        # Train DATER
        dater = DATER(max_iter=5)
        dater.fit(loader, train_idx, val_idx)
        dater_history.append(dater.iteration_history)

    # Generate visualizations
    plot_average_faces(loader)
    plot_iteration_curves(csa_history, dater_history)
    performance_df, summary_table = create_performance_table(csa_history, dater_history)

    # Print raw performance data
    print("\nRaw Performance Data:")
    print(performance_df)
    print("\nSummary Statistics:")
    print(summary_table)

if __name__ == "__main__":
    main()